In [20]:
from dataclasses import dataclass
from pathlib import Path
import os
from cnnClassifier.__init__ import logger
import zipfile
import gdown

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [13]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigManager:
    def __init__(self, config_file_path=str(CONFIG_FILE_PATH), params_file_path=str(PARAMS_FILE_PATH)):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        """
            To get the DataIngestionConfig object containing the essential directories and files
        """
        config_obj = self.config.data_ingestion 
        create_directories([config_obj.root_dir]) # created the root directory
    
        data_ingestion_config = DataIngestionConfig(       # Passing the essential params
            root_dir=config_obj.root_dir, 
            source_url=config_obj.source_url,
            local_data_file=config_obj.local_data_file,
            unzip_dir=config_obj.unzip_file
        )
    
        return data_ingestion_config
            

In [18]:
# Create a class for data ingestion from downloadintg the zip file from gdrive to unzipping it int right directory
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        # get the source url
        # path of file to store zip in
        # create the artifacts dir
        try:
            src_url = self.config.source_url
            zip_data_file_path = self.config.local_data_file
    
            os.makedirs("../artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading zip file from {src_url} into local file:- {zip_data_file_path}") # Log the info
    
            file_id = src_url.split('/')[-2]
            src_file_path = f"https://drive.google.com/uc?/export=download&id={file_id}"
            gdown.download(src_file_path, zip_data_file_path)
            logger.info(f"Downloaded the data set from {src_file_path} into the file:- {zip_data_file_path}")
        except Exception as e:
            raise e

        return zip_data_file_path


    def extract_zip_file(self):
        """
        Takes the zip file path: str , and unzip it into the data directory
        """
        unzip_data_dir = self.config.unzip_dir
        os.makedirs(unzip_data_dir, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_data_dir)
            

In [19]:
# Now run the entire pipline 
try:
    config = ConfigManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-02 15:59:29,899: INFO: common: common.py: 26: Loaded config from /home/codebind/Documents/pet_projects/Kidney-disease-Classification-/config/config.yaml successfully!!!!]
[2026-06-02 15:59:29,902: INFO: common: common.py: 26: Loaded config from /home/codebind/Documents/pet_projects/Kidney-disease-Classification-/params.yaml successfully!!!!]
[2026-06-02 15:59:29,904: INFO: common: common.py: 45: Created directory at :artifacts]
[2026-06-02 15:59:29,907: INFO: common: common.py: 45: Created directory at :artifacts/data_ingestion]
[2026-06-02 15:59:29,908: INFO: 3251286284: 3251286284.py: 15: Downloading zip file from https://drive.google.com/file/d/1sk4kp57TBwY6DeTX1b7lRQvcSyW8AyLk/view?usp=sharing into local file:- artifacts/data_ingestion/dataset.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1sk4kp57TBwY6DeTX1b7lRQvcSyW8AyLk
From (redirected): https://drive.google.com/uc?id=1sk4kp57TBwY6DeTX1b7lRQvcSyW8AyLk&confirm=t&uuid=56f24fba-5dca-4ae5-a566-28e31ddf26b1
To: /home/codebind/Documents/pet_projects/Kidney-disease-Classification-/notebooks/artifacts/data_ingestion/dataset.zip
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.63G/1.63G [02:45<00:00, 9.83MB/s]

[2026-06-02 16:02:18,631: INFO: 3251286284: 3251286284.py: 20: Downloaded the data set from https://drive.google.com/uc?/export=download&id=1sk4kp57TBwY6DeTX1b7lRQvcSyW8AyLk into the file:- artifacts/data_ingestion/dataset.zip]
